## 1. Conexão

> **Conexão protegida. É proibido alterar, substituir ou excluir as células de conexão.**


O estudo usa a sessão Spark existente, o DB2 V3 para leitura nativa e o HDFS V1 para a publicação dos arquivos.


In [ ]:
from traceback import format_exc

try:
    from src.utils.gerenciador_local_v2 import GerenciadorLocal

    gerenciador_local = GerenciadorLocal(
        nome_sessao="radar-todos-1",
        exibir_configuracao=False,
        ativar_logs=True,
    )

    spark = gerenciador_local.criar_sessao_spark(
        db2=True,
        driver_memory="16g",
        num_executors=6,
        executor_memory="4g",
        executor_cores=4,
        spark_conf={
            "spark.driver.memoryOverhead": "8g",
            "spark.executor.memoryOverhead": "2g",

            "spark.dynamicAllocation.enabled": "true",
            "spark.dynamicAllocation.minExecutors": "4",
            "spark.dynamicAllocation.initialExecutors": "6",
            "spark.dynamicAllocation.maxExecutors": "8",

            "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
            "spark.kryoserializer.buffer.max": "512m",

            "spark.speculation": "false",
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
            "spark.sql.adaptive.advisoryPartitionSizeInBytes": "64m",
            "spark.sql.adaptive.skewJoin.enabled": "true",
            "spark.sql.adaptive.localShuffleReader.enabled": "true",
            "spark.sql.shuffle.partitions": "320",

            "spark.sql.broadcastTimeout": "8000",
            "spark.executor.heartbeatInterval": "30s",
            "spark.network.timeout": "300s",
            "spark.sql.session.timeZone": "America/Sao_Paulo",
        },
    )

    try:
        widget_cls = __import__("ipywidgets").Widget
        ipython = get_ipython()
        if ipython is not None:
            ipython.display_formatter.formatters["text/plain"].for_type(
                widget_cls,
                lambda *a, **k: None,
            )
    except (ImportError, NameError, AttributeError):
        pass

    print("Sessão Spark inicializada.")

except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise


In [ ]:
%run ./src/utils/gerenciador_spark_v2.ipynb
%run ./src/utils/gerenciador_db2_spark_v3.ipynb
%run ./src/utils/gerenciador_hdfs_spark_v1.ipynb


## 2. Parâmetros da execução com pacote ZIP

Execute a partir da raiz do projeto. A primeira execução está configurada para homologação técnica. Com `somente_teste_reduzido = True`, o driver tria as 13 consultas, confere uma CTE de uma linha em SYSIBM.SYSDUMMY1 e publica o ZIP sintético HOMOLOGACAO_GFP.zip. Para testar a cópia no JupyterLab, habilite também `baixar_zip_jupyterlab` e confira o cluster. Após o aceite, configure `somente_teste_reduzido = False` para a extração.

Configure `data_inicial` (inclusiva) e `data_final_exclusiva` em AAAA-MM-DD. A janela inicial cobre sete dias, de 18 a 24/11/2026. Para estudar apenas 24/11, use `2026-11-24` e `2026-11-25`. A duração é livre. O último dia incluído pode ser hoje, segundo o calendário da sessão Spark. Nesse caso, os dados do dia ainda podem mudar.

As consultas principais extraem agregados de conta corrente e cartão em BRL, no estado 0, preservando os valores dos seis indicadores. A Q04 simula filtros sobre o universo original. A Q07 mede inclusões do cliente em outro universo e a Q08 investiga a cobertura dos derivados dos originais divididos. Q09 e Q10 complementam a investigação. Q12 fica fora do escopo.

O catálogo documenta a semântica e o código GFP V7 demonstra usos específicos na leitura. O notebook de referência do Radar distingue qualificação implementada de etapas transacionais planejadas. As extrações fornecem evidência empírica da janela. Inferências e hipóteses sobre atribuição e recomendações de filtros pertencem à análise posterior.


In [ ]:
from pathlib import Path

revisao_notebook = "2026-11-periodo-configuravel"
somente_teste_reduzido = True  # Após a homologação, use False para a extração das 13 consultas.
baixar_zip_jupyterlab = False
cluster_bbmagic = "CDP"  # Mesmo cluster da sessão protegida.

data_inicial = "2026-11-18"
data_final_exclusiva = "2026-11-25"
consultas_complementares = ("Q09", "Q10")
destino_hdfs = None  # Ou um diretório existente no HDFS da sessão.
timeout_transacoes_segundos = 7200
timeout_cadastros_segundos = 900

fonte_modulo = Path("src/utils/estudo_indicadores_gfp.py").read_bytes().decode("utf-8")
contexto_local_estudo = {}
exec(compile(fonte_modulo, "estudo_indicadores_gfp.py", "exec"), contexto_local_estudo)
consultas_estudo = contexto_local_estudo["preparar_consultas"](
    Path("03_CONSULTAS"), data_inicial, data_final_exclusiva, consultas_complementares,
)
configuracao_estudo = {
    "inicio": data_inicial,
    "fim": data_final_exclusiva,
    "destino": destino_hdfs,
    "timeout_transacoes": timeout_transacoes_segundos,
    "timeout_cadastros": timeout_cadastros_segundos,
}
fontes_estudo = {
    "ESTUDO_RADAR.ipynb": contexto_local_estudo["preparar_notebook"](Path.cwd(), revisao_notebook),
    "estudo_indicadores_gfp.py": fonte_modulo,
}
print("Consultas selecionadas", ", ".join(consultas_estudo))


## 3. Extrair, conferir e baixar o ZIP

O DB2 V3 executa as consultas avulsas em sequência na mesma conexão JDBC de leitura, com isolamento READ_COMMITTED e autocommit. As CTEs e os agrupamentos são executados no DB2. O HDFS V1 publica os resultados com permissões privadas e valida o ZIP antes de publicá-lo.

O limite JDBC é de duas horas por consulta transacional e 15 minutos para cada cadastro. O manifesto registra o tempo observado de cada tentativa.

Com destino_hdfs igual a None, a execução usa ESTUDO_INDICADORES_GFP/04_RESULTADOS_DB2 na home HDFS. Execute o notebook integralmente e baixe apenas o ZIP pelo caminho informado na saída. O pacote contém os CSVs disponíveis, os SQLs das tentativas, o manifesto, o notebook sem saídas gravadas e o módulo executado. A situação do pacote e o SHA-256 são exibidos após a validação. Em caso de falha parcial, o ZIP identifica as consultas com erro e os arquivos disponíveis.


In [ ]:
from uuid import uuid4

id_execucao_gfp = uuid4().hex
codigo_remoto_estudo = (
    f"recibo_zip_gfp = {{'execucao': {id_execucao_gfp!r}}}\n"
    f"fontes_estudo = {fontes_estudo!r}\n"
    "contexto_estudo = {}\n"
    "exec(compile(fontes_estudo['estudo_indicadores_gfp.py'], "
    "'estudo_indicadores_gfp.py', 'exec'), contexto_estudo)\n"
)
if somente_teste_reduzido:
    codigo_remoto_estudo += (
        f"contexto_estudo['validar_periodo']({configuracao_estudo['inicio']!r}, {configuracao_estudo['fim']!r}, spark)\n"
        "GerenciadorLogs('HOMOLOGACAO').info('Gerenciadores carregados.')\n"
        "conector_teste_gfp = criar_conector_db2_spark()\n"
        "try:\n"
        "    contexto_estudo['testar_integracao_reduzida']("
        f"conector_teste_gfp, {consultas_estudo!r}, {destino_hdfs!r}, "
        "criar_gerenciador_hdfs_spark, recibo_zip_gfp)\n"
        "except Exception as erro:\n"
        "    raise RuntimeError(contexto_estudo['diagnostico_erro'](conector_teste_gfp, erro)) from None\n"
    )
else:
    codigo_remoto_estudo += (
        "contexto_estudo['executar_estudo']("
        f"criar_conector_db2_spark(), {consultas_estudo!r}, {configuracao_estudo!r}, "
        "fontes_estudo, criar_gerenciador_hdfs_spark, recibo_zip_gfp)\n"
    )
get_ipython().run_cell_magic("spark", "", codigo_remoto_estudo)


## 4. Download opcional no JupyterLab

Ative `baixar_zip_jupyterlab` nos parâmetros para obter o ZIP desta execução na pasta do notebook. A célula recebe o recibo do driver e publica a cópia local após conferir o SHA-256. Se a execução parar depois de publicar um pacote parcial, execute somente esta célula.


In [ ]:
if baixar_zip_jupyterlab:
    import hashlib
    import os
    import re
    from tempfile import TemporaryDirectory
    from urllib.parse import urlsplit

    from bbmagic import Hdfs

    recibo_gfp_local = None
    get_ipython().run_cell_magic(
        "spark", "-o recibo_gfp_local -n 1",
        "recibo_gfp_local = spark.createDataFrame([recibo_zip_gfp], "
        "'execucao string, caminho string, sha256 string')",
    )
    if recibo_gfp_local is None or len(recibo_gfp_local) != 1:
        raise RuntimeError("Recibo indisponível no JupyterLab. Consulte o caminho publicado no HDFS.")
    recibo = recibo_gfp_local.iloc[0]
    if recibo["execucao"] != id_execucao_gfp or not re.fullmatch(r"[0-9a-f]{64}", str(recibo["sha256"])):
        raise RuntimeError("A execução atual não possui um ZIP validado para download.")
    uri_zip = urlsplit(recibo["caminho"])
    if uri_zip.scheme != "hdfs" or not uri_zip.netloc or not uri_zip.path.endswith(".zip"):
        raise ValueError("Caminho do ZIP inválido no recibo.")
    destino_zip = Path.cwd() / f"{id_execucao_gfp}_{Path(uri_zip.path).name}"
    if destino_zip.exists():
        raise FileExistsError(destino_zip.name)
    with TemporaryDirectory(dir=destino_zip.parent) as pasta_download:
        temporario = Path(pasta_download) / "pacote.zip"
        try:
            Hdfs(gerenciador_local.keytab, cluster=cluster_bbmagic).download(
                uri_zip.path, str(temporario), overwrite=False,
            )
        except Exception as erro:
            raise RuntimeError(f"Falha no download BBMagic ({type(erro).__name__}). O ZIP permanece no HDFS.") from None
        os.chmod(temporario, 0o600)
        digest = hashlib.sha256()
        with temporario.open("rb") as entrada:
            for bloco in iter(lambda: entrada.read(262144), b""):
                digest.update(bloco)
        if digest.hexdigest() != recibo["sha256"]:
            raise RuntimeError("SHA-256 divergente no download. O original permanece no HDFS.")
        os.link(temporario, destino_zip)  # Publicação exclusiva no mesmo filesystem do JupyterLab.
    print("ZIP no JupyterLab com SHA-256 conferido", destino_zip)
